# TA-005B — Optimización de Umbrales: DenseNet-121 Modelo Ganador
**Proyecto:** Sistema Multimodal de IA para Apoyo Diagnóstico Clínico Veterinario — Vargas Vet  
**Curso:** Taller Integrador 1 — UPAO  
**Integrantes:** Baylón Toledo, Diogho Matteo (PM) · Saavedra Arroyo, Sebastián Alonso (Scrum Master)  
**Sprint 2:** 31 May – 20 Jun 2026  

**Contexto:** El benchmarking (TA-004) demostró que DenseNet-121 con pesos ImageNet logró AUC=0.8717 en test (umbral O4 ≥ 0.80 cumplido). El F1-macro=0.5846 no alcanzó el umbral O4 (≥ 0.75) usando umbral fijo de 0.5 por clase. Este notebook aplica optimización de umbral por clase usando el validation set para mejorar el F1-macro en test.

**Objetivo:**
1. Cargar el mejor checkpoint de DenseNet-121 (TA-004)
2. Buscar el umbral óptimo por clase en validation set (maximizar F1)
3. Re-evaluar en test con umbrales optimizados
4. Guardar umbrales en JSON para uso en FastAPI
5. Verificar cumplimiento de umbrales O4 del Project Charter

## 1 · Verificación de entorno GPU

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print('Dispositivo activo:', DEVICE)

## 2 · Imports y configuración

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve, precision_recall_curve
from pathlib import Path
import json
import time
from tqdm import tqdm
from PIL import Image

sns.set_style('whitegrid')

BASE = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\Radiografias')
MANIFESTS     = BASE / 'dataset_split' / 'manifests'
DATASET_SPLIT = BASE / 'dataset_split'
CHECKPOINTS_DIR = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\modelos\checkpoints')
NOTEBOOKS_DIR   = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\modelos\Notebooks')

SEED = 42
IMAGE_SIZE = 224
CLASSES = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)
BATCH_SIZE = 16

np.random.seed(SEED)
torch.manual_seed(SEED)

print('Configuración:')
print(f'  Checkpoint a cargar: densenet_best.pth (TA-004)')
print(f'  Clases: {CLASSES}')
print(f'  Estrategia: umbral óptimo por clase en validation set')

## 3 · Cargar manifests

In [ ]:
df_train = pd.read_csv(MANIFESTS / 'train.csv')
df_val   = pd.read_csv(MANIFESTS / 'val.csv')
df_test  = pd.read_csv(MANIFESTS / 'test.csv')

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

## 4 · Dataset y dataloaders
> Sin augmentation — solo Resize y normalización para inferencia limpia.

In [ ]:
def load_dicom_normalized(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    p2, p98 = np.percentile(img, [2, 98])
    img = np.clip(img, p2, p98)
    img = (img - p2) / (p98 - p2 + 1e-8)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        img = 1.0 - img
    return (img * 255).astype(np.uint8)

def build_label_vector(tag_str, classes):
    vec = np.zeros(len(classes), dtype=np.float32)
    if pd.isna(tag_str):
        vec[-1] = 1.0
        return vec
    tags = [t.strip() for t in str(tag_str).split('|')]
    for i, cls in enumerate(classes):
        if cls in tags:
            vec[i] = 1.0
    if vec.sum() == 0:
        vec[-1] = 1.0
    return vec

class VetXRayDataset(Dataset):
    def __init__(self, df, base_path, classes, split='val'):
        self.df = df.reset_index(drop=True)
        self.base_path = Path(base_path)
        self.classes = classes
        self.split = split
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dcm_path = self.base_path / self.split / row['FileName']
        img_gray = load_dicom_normalized(dcm_path)
        img_pil = Image.fromarray(img_gray)
        img_rgb = Image.merge('RGB', [img_pil, img_pil, img_pil])
        img_tensor = self.transform(img_rgb)
        label = build_label_vector(row['TAG'], self.classes)
        return img_tensor, torch.from_numpy(label)

val_ds  = VetXRayDataset(df_val,  DATASET_SPLIT, CLASSES, split='val')
test_ds = VetXRayDataset(df_test, DATASET_SPLIT, CLASSES, split='test')

val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

img, lbl = val_ds[0]
print(f'Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Forma tensor: {img.shape}')

## 5 · Cargar modelo DenseNet-121 (checkpoint TA-004)
> Se carga el mejor checkpoint del benchmarking: AUC val=0.8630, AUC test=0.8717.

In [ ]:
model = models.densenet121(weights=None)
in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

ckpt_path = CHECKPOINTS_DIR / 'densenet_best.pth'
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print(f'Checkpoint cargado: {ckpt_path}')
print(f'Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')
print('Modelo listo para inferencia')

## 6 · Obtener probabilidades en Validation y Test
> Se obtienen los logits de ambos sets para calcular umbrales en val y evaluar en test.

In [ ]:
def get_probs_and_labels(model, loader, device):
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc='Inferencia', leave=False):
            imgs = imgs.to(device)
            logits = model(imgs)
            all_logits.append(logits.cpu().numpy())
            all_labels.append(lbls.numpy())
    logits = np.vstack(all_logits)
    labels = np.vstack(all_labels)
    probs  = 1 / (1 + np.exp(-logits))
    return probs, labels

print('Obteniendo probabilidades en validation...')
val_probs, val_labels = get_probs_and_labels(model, val_loader, DEVICE)

print('Obteniendo probabilidades en test...')
test_probs, test_labels = get_probs_and_labels(model, test_loader, DEVICE)

print(f'Val  — probs shape: {val_probs.shape}  | labels shape: {val_labels.shape}')
print(f'Test — probs shape: {test_probs.shape} | labels shape: {test_labels.shape}')

val_auc_base = roc_auc_score(val_labels, val_probs, average='macro')
val_f1_base  = f1_score(val_labels, (val_probs > 0.5).astype(int), average='macro', zero_division=0)
print(f'\nBaseline (umbral=0.5): AUC val={val_auc_base:.4f} | F1 val={val_f1_base:.4f}')

## 7 · Optimización de umbral por clase
> Para cada clase se busca el umbral que maximiza el F1-score en el validation set. Este umbral se usa luego en el test set y en FastAPI.

In [ ]:
def find_optimal_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_thresh = 0.5
    best_f1 = 0
    for t in thresholds:
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return round(float(best_thresh), 2), round(float(best_f1), 4)

optimal_thresholds = {}
print('Optimización de umbral por clase (validation set):')
print(f'{"Clase":<25} {"Umbral opt":>12} {"F1 con 0.5":>12} {"F1 óptimo":>12}')
print('-' * 65)

for i, cls in enumerate(CLASSES):
    thresh_opt, f1_opt = find_optimal_threshold(val_labels[:, i], val_probs[:, i])
    f1_base = f1_score(val_labels[:, i], (val_probs[:, i] >= 0.5).astype(int), zero_division=0)
    optimal_thresholds[cls] = thresh_opt
    print(f'{cls:<25} {thresh_opt:>12.2f} {f1_base:>12.4f} {f1_opt:>12.4f}')

print()
print('Umbrales óptimos:', optimal_thresholds)

## 8 · Evaluación en Test con umbrales optimizados

In [ ]:
thresh_vector = np.array([optimal_thresholds[cls] for cls in CLASSES])

test_preds_opt  = (test_probs >= thresh_vector).astype(int)
test_preds_base = (test_probs >= 0.5).astype(int)

test_auc  = roc_auc_score(test_labels, test_probs, average='macro')
test_f1_base = f1_score(test_labels, test_preds_base, average='macro', zero_division=0)
test_f1_opt  = f1_score(test_labels, test_preds_opt,  average='macro', zero_division=0)
test_acc_base = accuracy_score(test_labels.flatten(), test_preds_base.flatten())
test_acc_opt  = accuracy_score(test_labels.flatten(), test_preds_opt.flatten())

print('=== Resultados en TEST SET ===')
print(f'  AUC (macro):              {test_auc:.4f}  | >= 0.80: {test_auc >= 0.80}')
print(f'  F1-macro (umbral=0.5):    {test_f1_base:.4f}')
print(f'  F1-macro (umbral optimo): {test_f1_opt:.4f}  | >= 0.75: {test_f1_opt >= 0.75}')
print(f'  Accuracy (umbral=0.5):    {test_acc_base:.4f}')
print(f'  Accuracy (umbral optimo): {test_acc_opt:.4f}')
print()
mejora_f1 = (test_f1_opt - test_f1_base) * 100
print(f'  Mejora F1 por optimización: +{mejora_f1:.2f} pp')
print()
print(f'  Cumple O4 completo (AUC >= 0.80 Y F1 >= 0.75): {test_auc >= 0.80 and test_f1_opt >= 0.75}')

## 9 · F1 por clase — comparativa umbral fijo vs optimizado

In [ ]:
print(f'{"Clase":<25} {"Umbral":>8} {"F1 (0.5)":>10} {"F1 (opt)":>10} {"Mejora":>8}')
print('-' * 65)

for i, cls in enumerate(CLASSES):
    f1_base = f1_score(test_labels[:, i], test_preds_base[:, i], zero_division=0)
    f1_opt  = f1_score(test_labels[:, i], test_preds_opt[:, i],  zero_division=0)
    mejora  = (f1_opt - f1_base) * 100
    print(f'{cls:<25} {optimal_thresholds[cls]:>8.2f} {f1_base:>10.4f} {f1_opt:>10.4f} {mejora:>+7.2f}pp')

## 10 · Curvas ROC por clase

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, (cls, ax) in enumerate(zip(CLASSES, axes)):
    fpr, tpr, _ = roc_curve(test_labels[:, i], test_probs[:, i])
    auc_cls = roc_auc_score(test_labels[:, i], test_probs[:, i])
    ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC={auc_cls:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_title(cls.replace('_', '\n'), fontsize=8)
    ax.set_xlabel('FPR', fontsize=8)
    ax.set_ylabel('TPR', fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Curvas ROC por clase — DenseNet-121 Modelo Ganador (Test Set)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 11 · Tiempo de inferencia

In [ ]:
times = []
with torch.no_grad():
    for i, (imgs, _) in enumerate(test_loader):
        if i >= 2:
            break
        imgs = imgs.to(DEVICE)
        t0 = time.time()
        _ = model(imgs)
        times.append((time.time() - t0) / imgs.size(0))

infer_time = np.mean(times)
print(f'Tiempo de inferencia promedio: {infer_time:.4f} seg/img')
print(f'Cumple umbral < 6 seg: {infer_time < 6}')

## 12 · Tabla resumen final vs umbrales O4

In [ ]:
resumen = pd.DataFrame({
    'Métrica': ['AUC (macro) test', 'F1-macro test (umbral=0.5)', 'F1-macro test (umbral opt)', 'Accuracy test', 'Inferencia (seg/img)'],
    'Valor': [f'{test_auc:.4f}', f'{test_f1_base:.4f}', f'{test_f1_opt:.4f}', f'{test_acc_opt:.4f}', f'{infer_time:.4f}'],
    'Umbral O4': ['≥ 0.80', '≥ 0.75', '≥ 0.75', '—', '< 6 seg'],
    'Cumple': [
        'Sí' if test_auc >= 0.80 else 'No',
        'Sí' if test_f1_base >= 0.75 else 'No',
        'Sí' if test_f1_opt >= 0.75 else 'No',
        '—',
        'Sí' if infer_time < 6 else 'No'
    ]
})

print(resumen.to_string(index=False))

## 13 · Guardar umbrales y resultados finales en JSON
> Los umbrales óptimos se guardan para ser usados directamente en el endpoint FastAPI /predict/radiografia.

In [ ]:
results_final = {
    'model': 'densenet121',
    'checkpoint': 'densenet_best.pth',
    'strategy': 'optimal_threshold_per_class',
    'classes': CLASSES,
    'optimal_thresholds': optimal_thresholds,
    'default_threshold': 0.5,
    'val_auc': float(roc_auc_score(val_labels, val_probs, average='macro')),
    'test_auc': float(test_auc),
    'test_f1_macro_base': float(test_f1_base),
    'test_f1_macro_opt': float(test_f1_opt),
    'test_accuracy_opt': float(test_acc_opt),
    'inference_time_sec': float(infer_time),
    'cumple_O4_auc': bool(test_auc >= 0.80),
    'cumple_O4_f1': bool(test_f1_opt >= 0.75),
    'cumple_inferencia': bool(infer_time < 6)
}

out_path = NOTEBOOKS_DIR / 'modelo_final_results.json'
with open(out_path, 'w') as f:
    json.dump(results_final, f, indent=4)

print(f'Resultados guardados: {out_path}')
print()
print('Umbrales óptimos para FastAPI:')
for cls, t in optimal_thresholds.items():
    print(f'  {cls}: {t}')

---
## Resumen Final — TA-005B

| Criterio O4 | Umbral | Resultado | Estado |
|---|---|---|---|
| AUC en test | ≥ 0.80 | Ver celda 8 | Ver celda 8 |
| F1-macro (umbral optimizado) | ≥ 0.75 | Ver celda 8 | Ver celda 8 |
| Inferencia por imagen | < 6 seg | Ver celda 11 | Ver celda 11 |
| Umbrales por clase guardados | — | modelo_final_results.json | ✅ |
| Checkpoint listo para FastAPI | — | densenet_best.pth | ✅ |

**Próximo paso:** TA-006 — Endpoint FastAPI `/predict/radiografia` usando `densenet_best.pth` y `modelo_final_results.json`.